# Pertemuan 7 — Studi Kasus Terintegrasi: Analisis Big Data End-to-End (Materi)
**Kasus:** Laporan Kinerja Triwulan Jaringan Toko Retail (Magelang, Yogyakarta, Semarang, Solo)

### 1. Inisialisasi PySpark Menggunakan spark_init

In [1]:
from spark_init import spark, F, Window
print("Status: SparkSession aktif & terisolasi!")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/09 07:11:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


⚡ SparkSession siap! (Spark 3.5.9 | local[*])
Status: SparkSession aktif & terisolasi!


### 2. Menyiapkan Tiga Sumber Data Sintetis & Unggah ke HDFS
- Data toko/cabang (`data_toko.json`) — JSON Lines
- Data katalog produk (`data_produk.csv`) — CSV
- Data transaksi triwulan (`data_transaksi_triwulan.csv`) — CSV (5.000 baris)

In [2]:
import os
import json
import numpy as np
import pandas as pd

np.random.seed(77)

# SUMBER 1: Data toko/cabang (JSON Lines)
toko_list = ["Toko Magelang", "Toko Yogyakarta", "Toko Semarang", "Toko Solo"]
data_toko = [
    {"toko_id": i + 1, "nama_toko": t, "kota": t.split()[1], "manager": np.random.choice(["Rani", "Joko", "Sari", "Bayu"])}
    for i, t in enumerate(toko_list)
]
with open("data_toko.json", "w") as f:
    for t in data_toko:
        f.write(json.dumps(t) + "\n")

# SUMBER 2: Data produk (CSV)
data_produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}",
     "kategori": np.random.choice(["Elektronik", "Fashion", "Makanan"]),
     "harga": int(np.random.choice([50000, 100000, 200000]))}
    for i in range(1, 21)
]
pd.DataFrame(data_produk).to_csv("data_produk.csv", index=False)

# SUMBER 3: Data transaksi triwulan (CSV) — 5.000 baris
n = 5000
data_transaksi = pd.DataFrame({
    "trx_id": [f"T{i}" for i in range(n)],
    "toko_id": np.random.randint(1, 5, size=n),
    "product_id": np.random.randint(1, 21, size=n),
    "qty": np.random.randint(1, 6, size=n),
})
data_transaksi.to_csv("data_transaksi_triwulan.csv", index=False)

print(f"Dataset lokal siap: {len(data_toko)} toko, {len(data_produk)} produk, {len(data_transaksi):,} transaksi.")

Dataset lokal siap: 4 toko, 20 produk, 5,000 transaksi.


### Ingestion Langsung ke HDFS

In [3]:
!hdfs dfs -mkdir -p /user/mahasiswa/pertemuan7/raw
!hdfs dfs -put -f data_transaksi_triwulan.csv /user/mahasiswa/pertemuan7/raw/
!hdfs dfs -put -f data_toko.json /user/mahasiswa/pertemuan7/raw/
!hdfs dfs -put -f data_produk.csv /user/mahasiswa/pertemuan7/raw/
!hdfs dfs -ls /user/mahasiswa/pertemuan7/raw

Found 3 items
-rw-r--r--   1 kaky supergroup        599 2026-10-09 07:12 /user/mahasiswa/pertemuan7/raw/data_produk.csv
-rw-r--r--   1 kaky supergroup        332 2026-10-09 07:12 /user/mahasiswa/pertemuan7/raw/data_toko.json
-rw-r--r--   1 kaky supergroup      61655 2026-10-09 07:11 /user/mahasiswa/pertemuan7/raw/data_transaksi_triwulan.csv


### 3. Tahap EXTRACT — Membaca Langsung dari Klaster HDFS

In [4]:
# Membaca berkas langsung dari sistem penyimpanan terdistribusi HDFS
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_transaksi_triwulan.csv",
    header=True, inferSchema=True
)
df_toko = spark.read.json("hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_toko.json")
df_produk = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_produk.csv",
    header=True, inferSchema=True
)

print(f"Transaksi : {df_transaksi.count():,} baris")
print(f"Toko      : {df_toko.count()} cabang")
print(f"Produk    : {df_produk.count()} produk")

Transaksi : 5,000 baris


Toko      : 4 cabang


Produk    : 20 produk


### 4. Tahap TRANSFORM — Multi-Join & Hitung Total Penjualan

In [5]:
df_gabungan = df_transaksi \
    .join(df_toko, on="toko_id", how="inner") \
    .join(df_produk, on="product_id", how="inner") \
    .withColumn("total_penjualan", F.col("qty") * F.col("harga"))

print(f"Total baris setelah join: {df_gabungan.count():,} (Integritas data 100% utuh)")
df_gabungan.show(5)

Total baris setelah join: 5,000 (Integritas data 100% utuh)


+----------+-------+------+---+----------+-------+---------------+-----------+----------+------+---------------+
|product_id|toko_id|trx_id|qty|      kota|manager|      nama_toko|nama_produk|  kategori| harga|total_penjualan|
+----------+-------+------+---+----------+-------+---------------+-----------+----------+------+---------------+
|         7|      1|    T0|  5|  Magelang|   Bayu|  Toko Magelang|   Produk-7|   Makanan|200000|        1000000|
|         5|      2|    T1|  5|Yogyakarta|   Bayu|Toko Yogyakarta|   Produk-5|Elektronik|200000|        1000000|
|         2|      3|    T2|  2|  Semarang|   Rani|  Toko Semarang|   Produk-2|Elektronik|100000|         200000|
|         2|      2|    T3|  2|Yogyakarta|   Bayu|Toko Yogyakarta|   Produk-2|Elektronik|100000|         200000|
|        18|      4|    T4|  2|      Solo|   Rani|      Toko Solo|  Produk-18|Elektronik|100000|         200000|
+----------+-------+------+---+----------+-------+---------------+-----------+----------+------+

### 5. Tahap TRANSFORM Analitik — Window Function Top-3 Produk per Toko

In [6]:
ringkasan_produk_toko = df_gabungan.groupBy("nama_toko", "nama_produk").agg(
    F.sum("total_penjualan").alias("total_penjualan")
)

window_toko = Window.partitionBy("nama_toko").orderBy(F.col("total_penjualan").desc())
top3_produk = ringkasan_produk_toko \
    .withColumn("peringkat", F.row_number().over(window_toko)) \
    .filter(F.col("peringkat") <= 3)

top3_produk.orderBy("nama_toko", "peringkat").show(12)

+---------------+-----------+---------------+---------+
|      nama_toko|nama_produk|total_penjualan|peringkat|
+---------------+-----------+---------------+---------+
|  Toko Magelang|   Produk-7|       46000000|        1|
|  Toko Magelang|   Produk-5|       44400000|        2|
|  Toko Magelang|  Produk-14|       38400000|        3|
|  Toko Semarang|   Produk-8|       49200000|        1|
|  Toko Semarang|   Produk-7|       40400000|        2|
|  Toko Semarang|   Produk-5|       39000000|        3|
|      Toko Solo|  Produk-14|       38400000|        1|
|      Toko Solo|   Produk-7|       37200000|        2|
|      Toko Solo|   Produk-5|       33200000|        3|
|Toko Yogyakarta|  Produk-14|       48200000|        1|
|Toko Yogyakarta|   Produk-5|       37600000|        2|
|Toko Yogyakarta|   Produk-4|       37000000|        3|
+---------------+-----------+---------------+---------+



### 6. Tahap TRANSFORM Analitik — Spark SQL Kontribusi Manajer Cabang

In [7]:
df_gabungan.createOrReplaceTempView("gabungan")

kontribusi_manager = spark.sql("""
    SELECT kota, manager, SUM(total_penjualan) AS total_penjualan
    FROM gabungan
    GROUP BY kota, manager
    ORDER BY total_penjualan DESC
""")
kontribusi_manager.show()

+----------+-------+---------------+
|      kota|manager|total_penjualan|
+----------+-------+---------------+
|  Semarang|   Rani|      416250000|
|  Magelang|   Bayu|      408550000|
|Yogyakarta|   Bayu|      389150000|
|      Solo|   Rani|      369950000|
+----------+-------+---------------+



### 7. Tahap LOAD — Menyimpan ke HDFS Parquet Terpartisi Kategori

In [8]:
!hdfs dfs -mkdir -p /user/mahasiswa/pertemuan7/processed

df_gabungan.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/user/mahasiswa/pertemuan7/processed/laporan_triwulan"
)

print("Laporan berhasil disimpan ke HDFS dalam format Parquet terpartisi.")
!hdfs dfs -ls /user/mahasiswa/pertemuan7/processed/laporan_triwulan

Laporan berhasil disimpan ke HDFS dalam format Parquet terpartisi.


Found 4 items
-rw-r--r--   3 kaky supergroup          0 2026-10-09 07:13 /user/mahasiswa/pertemuan7/processed/laporan_triwulan/_SUCCESS
drwxr-xr-x   - kaky supergroup          0 2026-10-09 07:13 /user/mahasiswa/pertemuan7/processed/laporan_triwulan/kategori=Elektronik
drwxr-xr-x   - kaky supergroup          0 2026-10-09 07:13 /user/mahasiswa/pertemuan7/processed/laporan_triwulan/kategori=Fashion


drwxr-xr-x   - kaky supergroup          0 2026-10-09 07:13 /user/mahasiswa/pertemuan7/processed/laporan_triwulan/kategori=Makanan


### 8. Verifikasi Akhir — Ringkasan Eksekutif (Executive Summary)

In [9]:
df_final = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/pertemuan7/processed/laporan_triwulan")

ringkasan_eksekutif = df_final.groupBy("nama_toko", "kota", "manager").agg(
    F.sum("total_penjualan").alias("total_penjualan_triwulan"),
    F.count("trx_id").alias("jumlah_transaksi"),
    F.round(F.avg("total_penjualan"), 2).alias("rata_rata_nilai_transaksi")
).orderBy(F.col("total_penjualan_triwulan").desc())

print("=== RINGKASAN EKSEKUTIF KINERJA CABANG ===")
ringkasan_eksekutif.show()

=== RINGKASAN EKSEKUTIF KINERJA CABANG ===


+---------------+----------+-------+------------------------+----------------+-------------------------+
|      nama_toko|      kota|manager|total_penjualan_triwulan|jumlah_transaksi|rata_rata_nilai_transaksi|
+---------------+----------+-------+------------------------+----------------+-------------------------+
|  Toko Semarang|  Semarang|   Rani|               416250000|            1306|                318721.29|
|  Toko Magelang|  Magelang|   Bayu|               408550000|            1315|                310684.41|
|Toko Yogyakarta|Yogyakarta|   Bayu|               389150000|            1200|                324291.67|
|      Toko Solo|      Solo|   Rani|               369950000|            1179|                313782.87|
+---------------+----------+-------+------------------------+----------------+-------------------------+

